In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

X = pd.read_csv("../data/processed/X_train_processed.csv")

y = pd.read_csv("../data/processed/y_train_log.csv")
y = y.squeeze()

In [2]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [3]:
alphas = [0.01, 0.1, 1, 10, 50, 100]

ridge_results = []

for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    
    ridge_model.fit(X_train, y_train)
    
    y_val_pred = ridge_model.predict(X_val)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    r2 = r2_score(y_val, y_val_pred)
    
    ridge_results.append({
        "alpha": alpha,
        "RMSE_log": rmse,
        "R2": r2
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df.sort_values(by="RMSE_log")

,alpha,RMSE_log,R2
1,0.10,0.120909,0.921660
2,1.00,0.121067,0.921455
0,0.01,0.122233,0.919936
3,10.00,0.128511,0.911500
4,50.00,0.134985,0.902359
5,100.00,0.138752,0.896833


In [4]:
from sklearn.model_selection import cross_val_score

best_ridge = Ridge(alpha=0.1)

cv_scores = cross_val_score(
    best_ridge,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_scores = -cv_scores

print("Cross-validation RMSE scores:", cv_rmse_scores)
print("Mean CV RMSE:", cv_rmse_scores.mean())
print("Standard deviation:", cv_rmse_scores.std())

Cross-validation RMSE scores: [0.12016038 0.14671816 0.14753148 0.11418105 0.16541121]
Mean CV RMSE: 0.1388004564733135
Standard deviation: 0.018976937137693557


In [6]:
# Load processed test data
X_test = pd.read_csv("../data/processed/X_test_processed.csv")

# Train final Ridge model on the full training data
final_model = Ridge(alpha=0.1)

final_model.fit(X, y)

# Predict on Kaggle test data
test_log_predictions = final_model.predict(X_test)

# Convert predictions back from log scale to normal price scale
test_predictions = np.expm1(test_log_predictions)

test_predictions[:10]

array([121537.40798291, 159425.73412907, 182816.00642654, 197639.42292072,
       198335.59870574, 168484.64217999, 174840.06884172, 158748.63239837,
       193483.2409396 , 118060.03078612])

In [7]:
# Load original Kaggle test file to get Id column
test_original = pd.read_csv("../data/raw/test.csv")

submission = pd.DataFrame({
    "Id": test_original["Id"],
    "SalePrice": test_predictions
})

submission.head()

,Id,SalePrice
0,1461,121537.407983
1,1462,159425.734129
2,1463,182816.006427
3,1464,197639.422921
4,1465,198335.598706


In [8]:
submission.to_csv("../data/processed/submission.csv", index=False)

print("Submission file saved successfully!")
print(submission.shape)

Submission file saved successfully!
(1459, 2)


In [9]:
import os
import joblib

# Create models folder if it does not exist
os.makedirs("../models", exist_ok=True)

# Save the final trained model
joblib.dump(final_model, "../models/final_ridge_model.joblib")

print("Final model saved successfully!")

Final model saved successfully!


In [10]:
os.listdir("../models")

['final_ridge_model.joblib']